# Build a model class in a notebook

This notebook reuses the dataset idea from `external_dataset_class.ipynb`.

Flow:
1. Include a dataset class locally so the notebook is self-contained.
2. Train an image-only model (`h.train()`).
3. Train a multimodal model that uses image + light curve + mask via `prepare_inputs`.

## 1) Self-contained dataset class (from the dataset notebook pattern)

In [ ]:
import numpy as np
from hyrax.datasets import HyraxDataset


class NotebookSurveyDatasetWithLightCurves(HyraxDataset):
    def __init__(self, config, data_location=None, n_objects=96, min_len=20, max_len=100):
        rng = np.random.default_rng(19)
        self.images = rng.normal(size=(n_objects, 3, 32, 32)).astype(np.float32)
        self.labels = rng.integers(0, 5, size=n_objects, dtype=np.int64)
        lengths = rng.integers(min_len, max_len + 1, size=n_objects)
        self.light_curves = [rng.normal(size=int(n)).astype(np.float32) for n in lengths]
        super().__init__(config)

    def __len__(self):
        return len(self.images)

    def get_image(self, idx):
        return self.images[idx]

    def get_label(self, idx):
        return self.labels[idx]

    def get_light_curve(self, idx):
        return self.light_curves[idx]

    def get_object_id(self, idx):
        return f"obj-{idx:05d}"

    def collate(self, samples):
        images = np.stack([s["data"]["image"] for s in samples], axis=0).astype(np.float32)
        labels = np.array([s["data"]["label"] for s in samples], dtype=np.int64)
        curves = [s["data"]["light_curve"] for s in samples]
        max_len = max(len(c) for c in curves)

        light_curve = np.zeros((len(curves), max_len), dtype=np.float32)
        light_curve_mask = np.zeros((len(curves), max_len), dtype=np.float32)
        for i, c in enumerate(curves):
            n = len(c)
            light_curve[i, :n] = c
            light_curve_mask[i, :n] = 1.0

        object_ids = np.array([s["data"]["object_id"] for s in samples], dtype=str)

        return {
            "data": {
                "image": images,
                "label": labels,
                "light_curve": light_curve,
                "light_curve_mask": light_curve_mask,
                "object_id": object_ids,
            }
        }

## 2) Image-only model and minimal training run

In [ ]:
import torch
import torch.nn as nn
from hyrax.models.model_registry import hyrax_model


@hyrax_model
class ImageOnlyCNN(nn.Module):
    def __init__(self, config, data_sample=None):
        super().__init__()
        self.config = config
        image_sample, _ = data_sample
        c, h, w = image_sample.shape[1:]
        self.net = nn.Sequential(
            nn.Conv2d(c, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8 * h * w, 5),
        )

    def forward(self, image):
        return self.net(image)

    @staticmethod
    def prepare_inputs(data_dict):
        image = data_dict["data"]["image"]
        label = data_dict["data"]["label"]
        return image, label

    def infer_batch(self, batch):
        image, _ = batch
        return self.forward(image)

    def train_batch(self, batch):
        image, labels = batch
        self.optimizer.zero_grad()
        logits = self.forward(image)
        loss = self.criterion(logits, labels)
        loss.backward()
        self.optimizer.step()
        return {"loss": loss.item()}

In [ ]:
from hyrax import Hyrax

h = Hyrax()
h.set_config("model.name", "ImageOnlyCNN")
h.set_config("train.epochs", 1)
h.set_config("data_loader.batch_size", 8)
h.set_config("criterion.name", "torch.nn.CrossEntropyLoss")
h.set_config("optimizer.name", "torch.optim.Adam")
h.set_config("torch.optim.Adam", {"lr": 1e-3})

h.set_config("data_request", {
    "train": {
        "science": {
            "dataset_class": "NotebookSurveyDatasetWithLightCurves",
            "fields": ["image", "label", "object_id"],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
        }
    },
    "validate": {
        "science": {
            "dataset_class": "NotebookSurveyDatasetWithLightCurves",
            "fields": ["image", "label", "object_id"],
            "primary_id_field": "object_id",
            "split_fraction": 0.2,
        }
    },
})

_ = h.train()

## 3) Multimodal model (image + light curve + mask)

In [ ]:
@hyrax_model
class ImageAndLightCurveModel(nn.Module):
    def __init__(self, config, data_sample=None):
        super().__init__()
        self.config = config
        image, light_curve, light_curve_mask, _ = data_sample

        c, h, w = image.shape[1:]
        lc_dim = light_curve.shape[1]

        self.image_encoder = nn.Sequential(
            nn.Conv2d(c, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(8 * h * w, 32),
            nn.ReLU(),
        )

        self.lc_encoder = nn.Sequential(
            nn.Linear(lc_dim, 32),
            nn.ReLU(),
        )

        self.classifier = nn.Linear(64, 5)

    def forward(self, image, light_curve, light_curve_mask):
        # Apply mask so padded values do not contribute to light-curve branch
        masked_lc = light_curve * light_curve_mask
        image_feat = self.image_encoder(image)
        lc_feat = self.lc_encoder(masked_lc)
        return self.classifier(torch.cat([image_feat, lc_feat], dim=1))

    @staticmethod
    def prepare_inputs(data_dict):
        d = data_dict["data"]
        image = d["image"]
        light_curve = d["light_curve"]
        light_curve_mask = d["light_curve_mask"]
        label = d["label"]
        return image, light_curve, light_curve_mask, label

    def infer_batch(self, batch):
        image, light_curve, light_curve_mask, _ = batch
        return self.forward(image, light_curve, light_curve_mask)

    def train_batch(self, batch):
        image, light_curve, light_curve_mask, labels = batch
        self.optimizer.zero_grad()
        logits = self.forward(image, light_curve, light_curve_mask)
        loss = self.criterion(logits, labels)
        loss.backward()
        self.optimizer.step()
        return {"loss": loss.item()}

In [ ]:
h2 = Hyrax()
h2.set_config("model.name", "ImageAndLightCurveModel")
h2.set_config("train.epochs", 1)
h2.set_config("data_loader.batch_size", 8)
h2.set_config("criterion.name", "torch.nn.CrossEntropyLoss")
h2.set_config("optimizer.name", "torch.optim.Adam")
h2.set_config("torch.optim.Adam", {"lr": 1e-3})

h2.set_config("data_request", {
    "train": {
        "science": {
            "dataset_class": "NotebookSurveyDatasetWithLightCurves",
            "fields": ["image", "light_curve", "label", "object_id"],
            "primary_id_field": "object_id",
            "split_fraction": 0.8,
        }
    },
    "validate": {
        "science": {
            "dataset_class": "NotebookSurveyDatasetWithLightCurves",
            "fields": ["image", "light_curve", "label", "object_id"],
            "primary_id_field": "object_id",
            "split_fraction": 0.2,
        }
    },
})

_ = h2.train()

## 4) Next step: move model + dataset classes into your external package

After this works, copy both classes into your package and switch `dataset_class` and `model.name` to fully-qualified import paths.